# Chapter 17 – Recursive algorithms: sum, Merge Sort, binary search, maximum subarray (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_cpp.ipynb)

The algorithms with pseudo-code of the chapter, run on the examples of the notes: the **sum of an array** (iterative, by halving and by decrease by one), **Merge Sort** with `Merge`, the **recursive binary search** and the **maximum subarray** problem (brute force, divide and conquer, Kadane's algorithm). The output reproduces the traces and the numbers of the chapter (with indices from 0).

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

Each program is written to a file by the `%%writefile` cell and then compiled and run with `g++ -std=c++17 -O2`: run the cells in order, from top to bottom. No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `RecursiveSum` (PascalCase).

## 1. The sum of an array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: a loop that accumulates the partial sums. $S(n)=3n+4$, extra memory $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide and conquer on the two halves (base case $l=r$). $2n-1$ calls, $S(n)=7n-5$, depth $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decrease by one: $S(n)=3n-1$, depth $n$.

*Input*: an array of $n\ge1$ integers (for `RecursiveSum`, two indices $l\le r$). *Output*: the sum of the elements. The counters `steps` (lines of pseudo-code executed), `calls` and `maxDepth` (simultaneously active calls) measure the quantities analysed in the chapter; in `RecursiveSum` and `RecursiveSumLinear` there is a single `return` so that the depth can be updated.

On the array of the example $a=[9,8,12,3,5,14,6]$ ($n=7$) the sum is $57$; `RecursiveSum` makes $13$ calls and $44$ steps, with depth $4$. Then we run the trace of the table of the chapter on $a=[9,8,12,3,5]$: the column *stack* reports the active calls.

In [1]:
%%writefile sums.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // steps executed (lines of pseudo-code)
int calls = 0;         // calls executed
int depth = 0;         // calls active right now
int maxDepth = 0;      // maximum depth of the recursion

void Reset()
{
    steps = 0;
    calls = 0;
    depth = 0;
    maxDepth = 0;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Iterative sum of a[0..n-1]
int SumIterative(int n, int *a)
{
    int s = 0;
    int i = 0;
    steps += 2;                 // lines 1-2: s := 0, i := 1
    while (i < n)
    {
        steps += 3;             // true test + two lines of the body
        s = s + a[i];
        i = i + 1;
    }
    steps += 2;                 // false test + return
    return s;
}

// Sum of a[l..r] (0-based indices), l <= r, by divide and conquer
int RecursiveSum(int *a, int l, int r)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // line 1: if l = r
    int result;
    if (l == r)
    {
        steps++;                // line 2: return a[l] (base case)
        result = a[l];
    }
    else
    {
        steps += 4;             // lines 3-6: m, two calls, return
        int m = l + (r - l) / 2;                // divide
        int s1 = RecursiveSum(a, l, m);         // conquer (left half)
        int s2 = RecursiveSum(a, m + 1, r);     // conquer (right half)
        result = s1 + s2;                       // combine
    }
    depth--;
    return result;
}

// Sum of a[0..n-1] by decrease by one, n >= 1
int RecursiveSumLinear(int *a, int n)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // line 1: if n = 1
    int result;
    if (n == 1)
    {
        steps++;                // line 2: return a[1]
        result = a[0];
    }
    else
    {
        steps += 2;             // lines 3-4: call and return
        int s = RecursiveSumLinear(a, n - 1);
        result = s + a[n - 1];
    }
    depth--;
    return result;
}

// ---- trace of RecursiveSum ----
int row = 0;
int maxStack = 0;

string Join(const vector<string> &v)
{
    string s;
    for (size_t i = 0; i < v.size(); i++)
    {
        if (i > 0) s += ",";
        s += v[i];
    }
    return s;
}

void Emit(const vector<string> &stack, const string &action)
{
    row++;
    if ((int)stack.size() > maxStack) maxStack = stack.size();
    printf("%2d | %s | %s\n", row, Join(stack).c_str(), action.c_str());
}

string P(int l, int r) { return "(" + to_string(l) + "," + to_string(r) + ")"; }

int RecursiveSumTrace(int *a, int l, int r, vector<string> &stack)
{
    stack.push_back(P(l, r));
    if (l == r)
    {
        Emit(stack, "l=r => return a[" + to_string(l) + "]=" + to_string(a[l]));
        stack.pop_back();
        return a[l];
    }
    int m = l + (r - l) / 2;
    Emit(stack, "l!=r, m=" + to_string(l) + "+(" + to_string(r) + "-" + to_string(l) + ")/2=" + to_string(m) + "; call " + P(l, m));
    int s1 = RecursiveSumTrace(a, l, m, stack);
    Emit(stack, "s1=" + to_string(s1) + "; call " + P(m + 1, r));
    int s2 = RecursiveSumTrace(a, m + 1, r, stack);
    Emit(stack, "s2=" + to_string(s2) + "; return " + to_string(s1) + "+" + to_string(s2) + "=" + to_string(s1 + s2));
    stack.pop_back();
    return s1 + s2;
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 8, 12, 3, 5, 14, 6};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    Reset();
    int s = SumIterative(n, a);
    printf("SumIterative       = %d   steps=%lld (3n+4=%d)\n", s, steps, 3 * n + 4);
    Reset();
    s = RecursiveSum(a, 0, n - 1);
    printf("RecursiveSum       = %d   calls=%d (2n-1=%d)  steps=%lld (7n-5=%d)  maxDepth=%d\n", s, calls, 2 * n - 1, steps, 7 * n - 5, maxDepth);
    Reset();
    s = RecursiveSumLinear(a, n);
    printf("RecursiveSumLinear = %d   calls=%d  steps=%lld (3n-1=%d)  maxDepth=%d\n", s, calls, steps, 3 * n - 1, maxDepth);

    int b[] = {9, 8, 12, 3, 5};
    printf("\n");
    printf("a = %s   RecursiveSum(a, 0, 4)\n", Fmt(b, 5).c_str());
    printf(" # | stack | action\n");
    vector<string> stack;
    int res = RecursiveSumTrace(b, 0, 4, stack);
    printf("result %d, rows %d, maximum number of active calls %d\n", res, row, maxStack);
    return 0;
}
#endif

Writing sums.cpp


In [2]:
!g++ -std=c++17 -O2 -o sums sums.cpp && ./sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | action
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Try it yourself.** Change the array `a` (also its length) and compare steps, calls and depth: only `RecursiveSumLinear` has depth $n$.

In [3]:
%%writefile prova_sums.cpp
#define NOMAIN
#include "sums.cpp"

int main()
{
    // Change the array: compare steps, calls and depth of the three algorithms.
    int a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
    int n = 16;
    Reset(); int s = SumIterative(n, a);
    printf("%d %lld\n", s, steps);
    Reset(); s = RecursiveSum(a, 0, n - 1);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    Reset(); s = RecursiveSumLinear(a, n);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    return 0;
}

Writing prova_sums.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sums prova_sums.cpp && ./prova_sums

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort and Merge

- **MergeSort**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sub-array $a[l..r]$ sorted in non-decreasing order. It is called `MergeSort(a, 1, n)` in the pseudo-code, `merge_sort(a, 0, n-1)` in the code.
- **Merge**$(a,l,m,r)$ — *Input*: an array and indices $l\le m<r$ such that $a[l..m]$ and $a[m+1..r]$ are sorted. *Output*: $a[l..r]$ sorted. It copies the two halves into $L$ and $R$ and moves back into $a$, step by step, the smaller of the first two elements not yet copied.

The counter `comparisons` counts the comparisons between elements executed by `Merge`. On the array of the example $a=[9,12,3,16,19,11,12]$ we print the calls of `Merge` in execution order (with the array after each of them, table of the chapter; indices are from 0: adding 1 gives those of the chapter), then the trace of `Merge(a, 0, 1, 3)` on $a=[5,8,2,7]$ (example of the C++ implementation).

In [5]:
%%writefile merge_sort.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // comparisons between elements executed by Merge
int msCalls = 0;             // calls of MergeSort

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r): merges the sorted a[l..m] and a[m+1..r]
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;             // size of the left part
    int n2 = r - m;                 // size of the right part
    vector<int> L(n1), R(n2);       // auxiliary arrays
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)        // merge L and R into a[l..r]
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)                  // copy the rest of L
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)                  // copy the rest of R
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

// MergeSort(a, l, r): sorts a[l..r]
void MergeSort(int *a, int l, int r)
{
    msCalls++;
    if (l < r)
    {
        int m = l + (r - l) / 2;    // divide
        MergeSort(a, l, m);         // conquer (left half)
        MergeSort(a, m + 1, r);     // conquer (right half)
        Merge(a, l, m, r);          // combine
    }
}

// ---- variants that print the trace ----
void MergeSortLog(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSortLog(a, l, m);
        MergeSortLog(a, m + 1, r);
        string left = Fmt(a + l, m - l + 1), right = Fmt(a + m + 1, r - m);
        Merge(a, l, m, r);
        printf("Merge(a,%d,%d,%d)  %s and %s  ->  %s\n", l, m, r, left.c_str(), right.c_str(), Fmt(a, 7).c_str());
    }
}

void MergeTrace(int *a, int n, int l, int m, int r)
{
    int n1 = m - l + 1, n2 = r - m;
    vector<int> L(a + l, a + m + 1), R(a + m + 1, a + r + 1);
    printf("Merge(a, %d, %d, %d) on a = %s\n", l, m, r, Fmt(a, n).c_str());
    printf("n1=%d, n2=%d, L=%s, R=%s\n", n1, n2, Fmt(L.data(), n1).c_str(), Fmt(R.data(), n2).c_str());
    int i = 0, j = 0, k = l, it = 0;
    while (i < n1 && j < n2)
    {
        it++;
        char head[200];
        if (L[i] <= R[j])
        {
            snprintf(head, sizeof head, "%2d. loop 1: L[%d]=%d <= R[%d]=%d? Yes => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
            a[k] = L[i];
            i++;
        }
        else
        {
            snprintf(head, sizeof head, "%2d. loop 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
            a[k] = R[j];
            j++;
        }
        k++;
        printf("%s, i=%d, j=%d, k=%d; a=%s\n", head, i, j, k, Fmt(a, n).c_str());
    }
    while (i < n1)
    {
        a[k] = L[i];
        printf("    loop 2 (rest of L): a[%d] = L[%d] = %d; a=%s\n", k, i, L[i], Fmt(a, n).c_str());
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        printf("    loop 3 (rest of R): a[%d] = R[%d] = %d; a=%s\n", k, j, R[j], Fmt(a, n).c_str());
        j++;
        k++;
    }
    printf("result a = %s, comparisons = %d\n", Fmt(a, n).c_str(), it);
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 12, 3, 16, 19, 11, 12};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("Calls of Merge in execution order:\n");
    int copia[] = {9, 12, 3, 16, 19, 11, 12};
    MergeSortLog(copia, 0, n - 1);
    comparisons = 0;
    msCalls = 0;
    MergeSort(a, 0, n - 1);
    printf("sorted = %s   calls of MergeSort=%d (2n-1=%d)   comparisons=%lld\n", Fmt(a, n).c_str(), msCalls, 2 * n - 1, comparisons);
    printf("\n");
    int b[] = {5, 8, 2, 7};
    MergeTrace(b, 4, 0, 1, 3);
    return 0;
}
#endif

Writing merge_sort.cpp


In [6]:
!g++ -std=c++17 -O2 -o merge_sort merge_sort.cpp && ./merge_sort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Calls of Merge in execution order:
Merge(a,0,0,1)  [9] and [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] and [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] and [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] and [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] and [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] and [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
sorted = [3, 9, 11, 12, 12, 16, 19]   calls of MergeSort=13 (2n-1=13)   comparisons=14

Merge(a, 0, 1, 3) on a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. loop 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. loop 1: L[0]=5 <= R[1]=7? Yes => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. loop 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    loop 2 (rest of L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
result a = [2, 5, 7, 8], comparisons = 3


**Try it yourself.** Sort another array: the number of comparisons of `Merge` is at most $n\lceil\log_2 n\rceil$ and the calls of `MergeSort` are always $2n-1$.

In [7]:
%%writefile prova_merge_sort.cpp
#define NOMAIN
#include "merge_sort.cpp"

int main()
{
    // Sort another array and count the comparisons (at most n*ceil(log2 n)).
    int b[] = {38, 27, 43, 3, 9, 82, 10, 5};
    int n = 8;
    comparisons = 0;
    msCalls = 0;
    MergeSort(b, 0, n - 1);
    printf("%s %lld %d\n", Fmt(b, n).c_str(), comparisons, msCalls);
    return 0;
}

Writing prova_merge_sort.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_merge_sort prova_merge_sort.cpp && ./prova_merge_sort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: an array $a$ of natural numbers sorted in non-decreasing order, two indices $left$ and $right$ and a value $m$. *Output*: an index $i\in\{left,\dots,right\}$ with $a[i]=m$, or $-1$ if $m$ is not in $a[left..right]$. Every call makes **at most one** recursive call on (about) half of the interval: the call tree is a path, with at most $\lfloor\log_2 n\rfloor+2$ calls. To search the whole array we call `RecursiveBinarySearch(a, 0, n-1, m)`.

On the same array $a=[3,9,11,12,12,16,19]$ of the chapter we search $m=12,10,3,19$. For each search the table has one row per call ($left$, $right$, $mid$, $a[mid]$, action), as in the chapter but with indices from 0: adding 1 gives those of the chapter (for example $m=19$ is found at index 6, that is position 7).

In [9]:
%%writefile binary_search.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int rbsCalls = 0;   // calls executed

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Index of m in a[left..right] (sorted array), or -1
int RecursiveBinarySearch(int *a, int left, int right, int m)
{
    rbsCalls++;
    if (left > right)
    {
        return -1;                  // empty interval: not found
    }
    int mid = left + (right - left) / 2;
    if (a[mid] == m)
    {
        return mid;                 // found
    }
    if (a[mid] < m)
    {
        return RecursiveBinarySearch(a, mid + 1, right, m);    // right half
    }
    return RecursiveBinarySearch(a, left, mid - 1, m);         // left half
}

// As above, but prints one row per call (k = number of the call)
int RbsTrace(int *a, int left, int right, int m, int k)
{
    if (left > right)
    {
        printf("%4d %4d %5d %3s %6s left > right => not found, return -1\n", k, left, right, "-", "-");
        return -1;
    }
    int mid = left + (right - left) / 2;
    printf("%4d %4d %5d %3d %6d ", k, left, right, mid, a[mid]);
    if (a[mid] == m)
    {
        printf("a[mid] = m => found, return %d\n", mid);
        return mid;
    }
    if (a[mid] < m)
    {
        printf("a[mid] < m => call on left=%d, right=%d\n", mid + 1, right);
        return RbsTrace(a, mid + 1, right, m, k + 1);
    }
    printf("a[mid] > m => call on left=%d, right=%d\n", left, mid - 1);
    return RbsTrace(a, left, mid - 1, m, k + 1);
}

#ifndef NOMAIN
int main()
{
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    int ms[] = {12, 10, 3, 19};
    for (int m : ms)
    {
        printf("\n");
        printf("Search for m = %d\n", m);
        printf("%4s %4s %5s %3s %6s action\n", "call", "left", "right", "mid", "a[mid]");
        RbsTrace(a, 0, n - 1, m, 1);
    }
    printf("\n");
    for (int m : ms)
    {
        rbsCalls = 0;
        int r = RecursiveBinarySearch(a, 0, n - 1, m);
        printf("m=%-3d -> %2d (%2d with indices from 1)  calls=%d\n", m, r, r >= 0 ? r + 1 : -1, rbsCalls);
    }
    return 0;
}
#endif

Writing binary_search.cpp


In [10]:
!g++ -std=c++17 -O2 -o binary_search binary_search.cpp && ./binary_search

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Search for m = 12
call left right mid a[mid] action
   1    0     6   3     12 a[mid] = m => found, return 3

Search for m = 10
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] < m => call on left=2, right=2
   3    2     2   2     11 a[mid] > m => call on left=2, right=1
   4    2     1   -      - left > right => not found, return -1

Search for m = 3
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] > m => call on left=0, right=0
   3    0     0   0      3 a[mid] = m => found, return 0

Search for m = 19
call left right mid a[mid] action
   1    0     6   3     12 a[mid] < m => call on left=4, right=6
   2    4     6   5     16 a[mid] < m => call on left=6, right=6
   3    6     6   6     19 a[mid] = m => found, return 6

m=12  ->  3 ( 4 with indices from 1)  calls=1
m=10  -> -1 (

**Try it yourself.** Search another value (present or absent) and check that the calls are at most $\lfloor\log_2 n\rfloor+2$.

In [11]:
%%writefile prova_binary_search.cpp
#define NOMAIN
#include "binary_search.cpp"

int main()
{
    // Search another value, also an absent one: the calls are at most floor(log2 n) + 2.
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    rbsCalls = 0;
    int r = RecursiveBinarySearch(a, 0, n - 1, 16);
    printf("%d %d\n", r, rbsCalls);
    rbsCalls = 0;
    r = RecursiveBinarySearch(a, 0, n - 1, 100);
    printf("%d %d\n", r, rbsCalls);
    return 0;
}

Writing prova_binary_search.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_binary_search prova_binary_search.cpp && ./prova_binary_search

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. It examines all the $n(n+1)/2$ subarrays $a[l..r]$ (non-empty, of consecutive elements), computing the sums incrementally: $\Theta(n^2)$.

On the example $a=[-2,5,-1,3]$ of the chapter the table has one row for each subarray examined ($l$, $r$, sum, action; every row that does not improve shows the current $best$). Indices are from 0: the chapter finds $(2,4,7)$, that is $a[2..4]=[5,-1,3]$ with sum $7$, here $(1,3,7)$.

In [13]:
%%writefile maxsub_brute.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

struct Triple
{
    int i, j, sum;
};

// Returns (bl, br, best): a maximum-sum subarray of a[0..n-1]
Triple BruteForceMaximumSubarray(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    int l = 0;
    while (l < n)
    {
        int sum = 0;
        int r = l;
        while (r < n)
        {
            sum = sum + a[r];             // sum of a[l..r]
            if (sum > best)
            {
                best = sum;
                bl = l;
                br = r;
            }
            r = r + 1;
        }
        l = l + 1;
    }
    return {bl, br, best};
}

// As above, but prints one row for each subarray examined
Triple BruteForceTrace(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    for (int l = 0; l < n; l++)
    {
        int sum = 0;
        for (int r = l; r < n; r++)
        {
            sum = sum + a[r];
            if (sum > best)
            {
                best = sum;
                bl = l;
                br = r;
                printf("%3d %3d %5d  sum > best => best=%d, bl=%d, br=%d\n", l, r, sum, best, bl, br);
            }
            else
            {
                printf("%3d %3d %5d  sum <= best=%d\n", l, r, sum, best);
            }
        }
    }
    return {bl, br, best};
}

#ifndef NOMAIN
int main()
{
    int a[] = {-2, 5, -1, 3};
    int n = 4;
    printf("a = %s   n = %d   subarrays n(n+1)/2 = %d\n", Fmt(a, n).c_str(), n, n * (n + 1) / 2);
    printf("%3s %3s %5s  action\n", "l", "r", "sum");
    Triple t = BruteForceTrace(a, n);
    printf("result (bl, br, best) = (%d, %d, %d)   (%d, %d, %d with indices from 1)\n", t.i, t.j, t.sum, t.i + 1, t.j + 1, t.sum);
    Triple u = BruteForceMaximumSubarray(a, n);
    printf("BruteForceMaximumSubarray(a, n) = (%d, %d, %d)\n", u.i, u.j, u.sum);
    return 0;
}
#endif

Writing maxsub_brute.cpp


In [14]:
!g++ -std=c++17 -O2 -o maxsub_brute maxsub_brute.cpp && ./maxsub_brute

a = [-2, 5, -1, 3]   n = 4   subarrays n(n+1)/2 = 10
  l   r   sum  action
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
result (bl, br, best) = (1, 3, 7)   (2, 4, 7 with indices from 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Try it yourself.** Change the array (for example with only negative numbers): the maximum subarray is the single largest element.

In [15]:
%%writefile prova_maxsub_brute.cpp
#define NOMAIN
#include "maxsub_brute.cpp"

int main()
{
    // Try an array of only negative numbers: the maximum subarray is the single largest element.
    int b[] = {-3, -1, -2, -5};
    Triple t = BruteForceMaximumSubarray(b, 4);
    printf("(%d, %d, %d)\n", t.i, t.j, t.sum);
    return 0;
}

Writing prova_maxsub_brute.cpp


In [16]:
!g++ -std=c++17 -O2 -o prova_maxsub_brute prova_maxsub_brute.cpp && ./prova_maxsub_brute

(1, 1, -1)


## 5. FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [17]:
%%writefile maxsub_dc.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int fmsCalls = 0;   // calls of FindMaximumSubarray

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

string Tr(const Triple &t)
{
    return "(" + to_string(t.i) + ", " + to_string(t.j) + ", " + to_string(t.sum) + ")";
}

// Returns (maxleft, maxright, sum) of the best subarray crossing mid
Triple FindMaxCrossingSubarray(int *a, int l, int mid, int r)
{
    int leftsum = INT_MIN;                  // -infinity
    int sum = 0;
    int maxleft = mid;
    for (int i = mid; i >= l; i--)          // from mid down to l
    {
        sum = sum + a[i];                   // sum of a[i..mid]
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
    }
    int rightsum = INT_MIN;
    sum = 0;
    int maxright = mid + 1;
    for (int j = mid + 1; j <= r; j++)      // from mid+1 to r
    {
        sum = sum + a[j];                   // sum of a[mid+1..j]
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
    }
    return {maxleft, maxright, leftsum + rightsum};
}

// Returns (i, j, sum) of a maximum-sum subarray of a[l..r]
Triple FindMaximumSubarray(int *a, int l, int r)
{
    fmsCalls++;
    if (l == r)
    {
        return {l, r, a[l]};                // base case: one element
    }
    int mid = l + (r - l) / 2;
    Triple left = FindMaximumSubarray(a, l, mid);
    Triple right = FindMaximumSubarray(a, mid + 1, r);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    if (left.sum >= right.sum && left.sum >= cross.sum)
    {
        return left;
    }
    if (right.sum >= left.sum && right.sum >= cross.sum)
    {
        return right;
    }
    return cross;
}

// ---- variants that print the trace ----
Triple Best(const Triple &left, const Triple &right, const Triple &cross)
{
    if (left.sum >= right.sum && left.sum >= cross.sum) return left;
    if (right.sum >= left.sum && right.sum >= cross.sum) return right;
    return cross;
}

Triple FmsTrace(int *a, int l, int r, int depth)
{
    string pad(2 * depth, ' ');
    if (l == r)
    {
        printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  base case\n", pad.c_str(), l, r, l, r, a[l]);
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d\n", pad.c_str(), l, r, mid);
    Triple left = FmsTrace(a, l, mid, depth + 1);
    Triple right = FmsTrace(a, mid + 1, r, depth + 1);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s\n", pad.c_str(), l, mid, r, Tr(cross).c_str());
    Triple ret = Best(left, right, cross);
    printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => returns %s\n", pad.c_str(), left.sum, right.sum, cross.sum, Tr(ret).c_str());
    return ret;
}

struct Row
{
    int l, r, mid;
    Triple left, right, cross, ret;
};

Triple FmsRows(int *a, int l, int r, vector<Row> &rows)
{
    if (l == r)
    {
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    Triple left = FmsRows(a, l, mid, rows);
    Triple right = FmsRows(a, mid + 1, r, rows);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    Triple ret = Best(left, right, cross);
    rows.push_back({l, r, mid, left, right, cross, ret});
    return ret;
}

void CrossingTrace(int *a, int l, int mid, int r)
{
    printf("FindMaxCrossingSubarray(a, %d, %d, %d)\n", l, mid, r);
    printf("%3s %5s %5s %8s %8s\n", "i", "a[i]", "sum", "leftsum", "maxleft");
    int leftsum = INT_MIN, maxleft = mid, sum = 0;
    for (int i = mid; i >= l; i--)
    {
        sum = sum + a[i];
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
        printf("%3d %5d %5d %8d %8d\n", i, a[i], sum, leftsum, maxleft);
    }
    printf("%3s %5s %5s %8s %8s\n", "j", "a[j]", "sum", "rightsum", "maxright");
    int rightsum = INT_MIN, maxright = mid + 1;
    sum = 0;
    for (int j = mid + 1; j <= r; j++)
    {
        sum = sum + a[j];
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
        printf("%3d %5d %5d %8d %8d\n", j, a[j], sum, rightsum, maxright);
    }
    printf("crossing subarray a[%d..%d], sum %d + %d = %d\n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
}

#ifndef NOMAIN
int main()
{
    int a[] = {-2, 5, -1, 3};
    printf("a = %s\n", Fmt(a, 4).c_str());
    Triple res = FmsTrace(a, 0, 3, 0);
    fmsCalls = 0;
    FindMaximumSubarray(a, 0, 3);
    printf("result %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)\n", Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * 4 - 1);

    printf("\n");
    int b[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(b, n).c_str());
    fmsCalls = 0;
    res = FindMaximumSubarray(b, 0, n - 1);
    printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)\n", n - 1, Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * n - 1);
    vector<Row> rows;
    FmsRows(b, 0, n - 1, rows);
    sort(rows.begin(), rows.end(), [](const Row &x, const Row &y)
         { return (x.r - x.l) != (y.r - y.l) ? (x.r - x.l) > (y.r - y.l) : x.l < y.l; });
    printf("%-9s %3s  %-13s %-13s %-13s returned\n", "call", "mid", "left", "right", "crossing");
    for (const Row &w : rows)
    {
        string call = "[" + to_string(w.l) + ".." + to_string(w.r) + "]";
        printf("%-9s %3d  %-13s %-13s %-13s %s\n", call.c_str(), w.mid, Tr(w.left).c_str(), Tr(w.right).c_str(), Tr(w.cross).c_str(), Tr(w.ret).c_str());
    }
    printf("\n");
    CrossingTrace(b, 0, 3, 7);
    printf("\n");
    CrossingTrace(b, 8, 11, 15);
    return 0;
}
#endif

Writing maxsub_dc.cpp


In [18]:
!g++ -std=c++17 -O2 -o maxsub_dc maxsub_dc.cpp && ./maxsub_dc

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [19]:
%%writefile prova_maxsub_dc.cpp
#define NOMAIN
#include "maxsub_dc.cpp"

int main()
{
    // Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
    int b[] = {4, -1, 2, 1, -5, 4, -7, 3, 3};
    fmsCalls = 0;
    Triple t = FindMaximumSubarray(b, 0, 8);
    printf("%s %d\n", Tr(t).c_str(), fmsCalls);
    return 0;
}

Writing prova_maxsub_dc.cpp


In [20]:
!g++ -std=c++17 -O2 -o prova_maxsub_dc prova_maxsub_dc.cpp && ./prova_maxsub_dc

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. A single scan: $cur$ is the maximum sum of a subarray **ending** at the current position and $cl$ its first index; if $cur<0$ we restart from $a[i]$, otherwise we extend. $\Theta(n)$ steps, $\Theta(1)$ extra memory. (Chapter 18 has the compact version that returns only the sum.)

On the stock example (daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) we print the table of the chapter: one row per iteration, with the values at the end of the iteration (indices from 0, row $i=0$ is the initialization). The result $(7,10,43)$, that is $(8,11,43)$ with indices from 1, coincides with the one of the divide and conquer algorithm.

In [21]:
%%writefile kadane.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Returns (bl, br, best): a maximum-sum subarray of a[0..n-1]
Triple Kadane(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    int cur = a[0];             // maximum sum of a subarray ending at i
    int cl = 0;
    int i = 1;
    while (i < n)
    {
        if (cur < 0)
        {
            cur = a[i];         // restart
            cl = i;
        }
        else
        {
            cur = cur + a[i];   // extend
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        i = i + 1;
    }
    return {bl, br, best};
}

// As above, but prints one row per iteration
Triple KadaneTrace(int *a, int n)
{
    int best = a[0], bl = 0, br = 0;
    int cur = a[0], cl = 0;
    printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", 0, a[0], "initialization", cur, cl, best, bl, br);
    for (int i = 1; i < n; i++)
    {
        const char *action;
        if (cur < 0)
        {
            cur = a[i];
            cl = i;
            action = "restart (cur<0)";
        }
        else
        {
            cur = cur + a[i];
            action = "extend";
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", i, a[i], action, cur, cl, best, bl, br);
    }
    return {bl, br, best};
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(a, n).c_str());
    printf("%3s %5s  %-24s %4s %3s %5s  (bl, br)\n", "i", "a[i]", "action", "cur", "cl", "best");
    Triple res = KadaneTrace(a, n);
    printf("result (%d, %d, %d)   (%d, %d, %d with indices from 1)\n", res.i, res.j, res.sum, res.i + 1, res.j + 1, res.sum);
    int b[] = {-2, 5, -1, 3};
    Triple t = Kadane(b, 4);
    printf("a = %s  Kadane(a, 4) = (%d, %d, %d)\n", Fmt(b, 4).c_str(), t.i, t.j, t.sum);
    return 0;
}
#endif

Writing kadane.cpp


In [22]:
!g++ -std=c++17 -O2 -o kadane kadane.cpp && ./kadane

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  action                    cur  cl  best  (bl, br)
  0    13  initialization             13   0    13  (0, 0)
  1    -3  extend                     10   0    13  (0, 0)
  2   -25  extend                    -15   0    13  (0, 0)
  3    20  restart (cur<0)            20   3    20  (3, 3)
  4    -3  extend                     17   3    20  (3, 3)
  5   -16  extend                      1   3    20  (3, 3)
  6   -23  extend                    -22   3    20  (3, 3)
  7    18  restart (cur<0)            18   7    20  (3, 3)
  8    20  extend                     38   7    38  (7, 8)
  9    -7  extend                     31   7    38  (7, 8)
 10    12  extend                     43   7    43  (7, 10)
 11    -5  extend                     38   7    43  (7, 10)
 12   -22  extend                     16   7    43  (7, 10)
 13    15  extend                     31   7    43  (7, 10)
 14    -4  extend                    

**Try it yourself.** Change the array (for example with only negative numbers): Kadane restarts at every step and finds the largest element.

In [23]:
%%writefile prova_kadane.cpp
#define NOMAIN
#include "kadane.cpp"

int main()
{
    // Try an array of only negative numbers: Kadane restarts at every step and finds the largest element.
    int c[] = {-3, -1, -2, -5};
    Triple t = Kadane(c, 4);
    printf("(%d, %d, %d)\n", t.i, t.j, t.sum);
    return 0;
}

Writing prova_kadane.cpp


In [24]:
!g++ -std=c++17 -O2 -o prova_kadane prova_kadane.cpp && ./prova_kadane

(1, 1, -1)
